In [1]:
import warnings
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import scipy.sparse

warnings.filterwarnings("ignore")

2025-03-26 18:18:38.180637: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-26 18:18:38.220469: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-26 18:18:38.220504: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-26 18:18:38.221588: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-26 18:18:38.228224: I tensorflow/core/platform/cpu_feature_guar

In [2]:
from preprocessing import preprocess_data
from basic_models import random_forest_classifier
from basic_models import gradient_boosting_classifier
from basic_models import LSTM_preds
from basic_models import logistic_reg_model

In [3]:
data, X_train_processed, X_test_processed, y_train, y_test = preprocess_data("stores_sales_forecasting.csv")
data

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,lag_profit_1,rolling_sales_mean_3,rolling_profit_mean_3,cohort,churn,Order Year,Order Month,Ship Year,Ship Month,Shipping Delay
20,79,US-2014-147606,2014-11-26,2014-12-01,Second Class,JE-15745,Joel Eaton,Consumer,United States,Houston,...,1.2130,316.092000,-42.551067,0,0,2014,11,2014,12,5
39,178,US-2015-101511,2015-11-21,2015-11-23,Second Class,JE-15745,Joel Eaton,Consumer,United States,Newark,...,-14.4750,171.047333,-8.199733,12,0,2015,11,2015,11,2
51,235,US-2017-100930,2017-04-07,2017-04-12,Standard Class,CS-12400,Christopher Schild,Home Office,United States,Tampa,...,-248.2458,370.848833,-116.764600,0,1,2017,4,2017,4,5
54,242,CA-2016-157749,2016-06-04,2016-06-09,Second Class,KL-16645,Ken Lonsdale,Consumer,United States,Chicago,...,-4.6752,202.864333,-160.638733,23,1,2016,6,2016,6,5
55,243,CA-2016-157749,2016-06-04,2016-06-09,Second Class,KL-16645,Ken Lonsdale,Consumer,United States,Chicago,...,-120.5130,64.319000,-42.673000,23,1,2016,6,2016,6,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2116,9963,CA-2015-168088,2015-03-19,2015-03-22,First Class,CM-12655,Corinna Mitchell,Home Office,United States,Houston,...,88.7332,316.008533,12.558933,0,0,2015,3,2015,3,3
2117,9965,CA-2016-146374,2016-12-05,2016-12-10,Second Class,HE-14800,Harold Engle,Corporate,United States,Newark,...,22.9885,51.110000,12.871967,0,0,2016,12,2016,12,5
2118,9981,US-2015-151435,2015-09-06,2015-09-09,Second Class,SW-20455,Shaun Weien,Consumer,United States,Lafayette,...,22.5296,106.393333,15.686000,0,1,2015,9,2015,9,3
2119,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,41.8608,340.516000,86.550800,0,1,2014,1,2014,1,2


In [4]:
rf_out = random_forest_classifier(X_train_processed, y_train)

In [5]:
gb_out = gradient_boosting_classifier(X_train_processed, y_train)

In [6]:
lr_out = logistic_reg_model(X_train_processed, y_train, X_test_processed)

In [7]:
X_train_dense = X_train_processed.toarray() if scipy.sparse.issparse(X_train_processed) else X_train_processed
X_test_dense = X_test_processed.toarray() if scipy.sparse.issparse(X_test_processed) else X_test_processed

lstm_out = LSTM_preds(X_train_dense, X_test_dense, y_train)

2025-03-26 18:18:42.914418: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9804 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:b1:00.0, compute capability: 7.5


Epoch 1/20


2025-03-26 18:18:46.151542: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
2025-03-26 18:18:46.971496: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fc79189e820 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-03-26 18:18:46.971528: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-03-26 18:18:46.976774: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1743013127.167858     721 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


11/11 [==============================] - 4s 15ms/step - loss: 0.5738 - accuracy: 0.0000e+00
Epoch 2/20
11/11 [==============================] - 0s 15ms/step - loss: 0.0723 - accuracy: 0.0000e+00
Epoch 3/20
11/11 [==============================] - 0s 15ms/step - loss: -0.4384 - accuracy: 0.0000e+00
Epoch 4/20
11/11 [==============================] - 0s 16ms/step - loss: -0.8632 - accuracy: 0.0000e+00
Epoch 5/20
11/11 [==============================] - 0s 14ms/step - loss: -1.2742 - accuracy: 0.0000e+00
Epoch 6/20
11/11 [==============================] - 0s 15ms/step - loss: -1.5640 - accuracy: 0.0000e+00
Epoch 7/20
11/11 [==============================] - 0s 15ms/step - loss: -1.9251 - accuracy: 0.0000e+00
Epoch 8/20
11/11 [==============================] - 0s 15ms/step - loss: -2.3189 - accuracy: 0.0000e+00
Epoch 9/20
11/11 [==============================] - 0s 15ms/step - loss: -2.6439 - accuracy: 0.0000e+00
Epoch 10/20
11/11 [==============================] - 0s 15ms/step - loss: -2.

In [8]:
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [9]:
# Define the kernel: Constant * RBF
kernel = C(1.0) * RBF(length_scale=1.0)

gp_model = GaussianProcessClassifier(kernel=kernel, random_state=42)

In [10]:
gp_model.fit(X_train_dense, y_train)

GaussianProcessClassifier(kernel=1**2 * RBF(length_scale=1), random_state=42)

In [11]:
y_pred = gp_model.predict(X_test_dense)

In [12]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7126


In [13]:
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Define the parameter grid for the kernel
kernel = C(1.0) * RBF(length_scale=1.0)

# Create the GaussianProcessClassifier
gp_model = GaussianProcessClassifier(kernel=kernel, random_state=42)

# Define the parameter grid to search over (remove alpha and tune other parameters)
param_grid = {
    'kernel__k1__constant_value': [1.0, 10.0, 100.0],  # Test different constant values for the kernel
    'kernel__k2__length_scale': [0.1, 1.0, 10.0],      # Test different length scales for the RBF kernel
    'n_restarts_optimizer': [0, 5, 10],                 # Number of restarts of the optimizer
    'max_iter_predict': [100, 200, 300],                # Test different iterations for prediction
    'optimizer': ['fmin_l_bfgs_b', 'fmin_tnc', 'fmin_powell'],  # Optimizer methods for kernel fitting
}

# Setup GridSearchCV with cross-validation
grid_search = GridSearchCV(estimator=gp_model, param_grid=param_grid, 
                           cv=5, n_jobs=-1, scoring='accuracy')

# Fit the grid search model
grid_search.fit(X_train_dense, y_train)

# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(f"Best hyperparameters found: {best_params}")

# Use the best model found by grid search
best_gp_model = grid_search.best_estimator_

# Make predictions with the best model
y_pred = best_gp_model.predict(X_test_dense)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("ROC AUC Score: ", roc_auc_score(y_test, y_pred))


/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: Converg

Best hyperparameters found: {'kernel__k1__constant_value': 1.0, 'kernel__k2__length_scale': 0.1, 'max_iter_predict': 100, 'n_restarts_optimizer': 5, 'optimizer': 'fmin_l_bfgs_b'}
Accuracy: 0.7126
Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.47      0.53        59
           1       0.76      0.83      0.79       115

    accuracy                           0.71       174
   macro avg       0.68      0.65      0.66       174
weighted avg       0.70      0.71      0.70       174

ROC AUC Score:  0.6546794399410464


In [14]:
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern, RationalQuadratic
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Define the parameter grid for testing different kernels
param_grid = {
    'kernel': [
        C(1.0) * RBF(length_scale=1.0),          # RBF Kernel
        C(1.0) * Matern(length_scale=1.0, nu=1.5),  # Matern Kernel
        C(1.0) * RationalQuadratic(length_scale=1.0, alpha=0.1),  # Rational Quadratic Kernel
    ],
    'n_restarts_optimizer': [0, 5, 10],          # Number of restarts of the optimizer
    'max_iter_predict': [100, 200, 300],         # Number of iterations for prediction
    'optimizer': ['fmin_l_bfgs_b', 'fmin_tnc', 'fmin_powell'],  # Optimizer methods
}

# Create the GaussianProcessClassifier
gp_model = GaussianProcessClassifier(random_state=42)

# Setup GridSearchCV with cross-validation
grid_search = GridSearchCV(estimator=gp_model, param_grid=param_grid, 
                           cv=5, n_jobs=-1, scoring='accuracy')

# Fit the grid search model
grid_search.fit(X_train_dense, y_train)

# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(f"Best hyperparameters found: {best_params}")

# Use the best model found by grid search
best_gp_model = grid_search.best_estimator_

# Make predictions with the best model
y_pred = best_gp_model.predict(X_test_dense)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("ROC AUC Score: ", roc_auc_score(y_test, y_pred))


/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:455: Converg

Best hyperparameters found: {'kernel': 1**2 * RBF(length_scale=1), 'max_iter_predict': 100, 'n_restarts_optimizer': 0, 'optimizer': 'fmin_l_bfgs_b'}
Accuracy: 0.7126
Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.47      0.53        59
           1       0.76      0.83      0.79       115

    accuracy                           0.71       174
   macro avg       0.68      0.65      0.66       174
weighted avg       0.70      0.71      0.70       174

ROC AUC Score:  0.6546794399410464


In [15]:
best_gp_model.predict(X_test_dense)

array([0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1,
       1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1])

In [26]:
X_test

NameError: name 'X_test' is not defined

In [17]:
df = pd.read_csv("stores_sales_forecasting.csv", encoding='ISO-8859-1')

In [18]:
df['predicted_churn'] = best_gp_model.predict(X_test_dense)

ValueError: Length of values (174) does not match length of index (2121)

In [33]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [36]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
latest_order_date = df['Order Date'].max()
df['churn'] = np.where((latest_order_date - df.groupby('Customer ID')['Order Date'].transform('max')).dt.days > 90, 1, 0)
X = df.drop(['churn', 'Row ID', 'Order ID', 'Customer Name', 'Product ID', 'Order Date', 'Ship Date',], axis=1)
y = df['churn']
        

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
X_test

,Ship Mode,Customer ID,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1845,Second Class,AB-10165,Consumer,United States,Lakewood,California,90712,West,Furniture,Furnishings,Flat Face Poster Frame,94.200,5,0.00,39.5640
1486,First Class,LW-16990,Corporate,United States,Henderson,Nevada,89015,West,Furniture,Tables,Global Adaptabilities Conference Tables,1685.880,6,0.00,320.3172
289,Same Day,PP-18955,Home Office,United States,Smyrna,Georgia,30080,South,Furniture,Furnishings,"Eldon 200 Class Desk Accessories, Black",18.840,3,0.00,7.1592
1607,Standard Class,DB-13210,Consumer,United States,Philadelphia,Pennsylvania,19134,East,Furniture,Furnishings,"6"" Cubicle Wall Clock, Black",58.248,9,0.20,11.6496
1857,Second Class,AW-10930,Home Office,United States,Houston,Texas,77070,Central,Furniture,Furnishings,Dax Clear Box Frame,6.984,2,0.60,-4.5396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1517,Second Class,TP-21415,Consumer,United States,Philadelphia,Pennsylvania,19134,East,Furniture,Chairs,HON 5400 Series Task Chairs for Big and Tall,4416.174,9,0.30,-630.8820
218,Second Class,MO-17500,Consumer,United States,New York City,New York,10035,East,Furniture,Bookcases,"Atlantic Metals Mobile 5-Shelf Bookcases, Cust...",722.352,3,0.20,90.2940
579,Standard Class,KN-16450,Corporate,United States,Redondo Beach,California,90278,West,Furniture,Bookcases,O'Sullivan Living Dimensions 2-Shelf Bookcases,308.499,3,0.15,-18.1470
2086,Standard Class,DL-12865,Consumer,United States,Long Beach,California,90805,West,Furniture,Chairs,"Global Airflow Leather Mesh Back Chair, Black",483.136,4,0.20,60.3920


In [ ]:
data, X_train_processed, X_test_processed, y_train, y_test

In [42]:
df_X_test = pd.DataFrame(X_test_dense, columns=original_feature_names)  

NameError: name 'original_feature_names' is not defined

In [41]:
len(X_test_dense)

174

In [39]:
X_test['predicted_churn'] = best_gp_model.predict(X_test_dense)


ValueError: Length of values (174) does not match length of index (425)

In [45]:
# One-Hot Encoding with pandas get_dummies
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)


In [47]:
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import StackingClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Gaussian Process
gp_model = GaussianProcessClassifier(kernel=C(1.0) * RBF(length_scale=1.0), random_state=42)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Logistic Regression
log_reg_model = LogisticRegression(random_state=42)

# Meta-model (stacked model) - Usually Logistic Regression
stacked_model = StackingClassifier(
    estimators=[('gp', gp_model), ('rf', rf_model), ('lr', log_reg_model)],
    final_estimator=LogisticRegression()
)

# Train the stacked model
stacked_model.fit(X_train, y_train)

# Make predictions using the stacked model
y_pred = stacked_model.predict(X_test)

# Evaluate the performance of the stacked model
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacked Model Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- City_Apopka
- City_Athens
- City_Auburn
- City_Bangor
- City_Bowling Green
- ...
Feature names seen at fit time, yet now missing:
- City_Allen
- City_Allentown
- City_Amarillo
- City_Andover
- City_Apple Valley
- ...


In [48]:
# One-Hot Encoding (if needed)
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)


In [49]:
# Align X_test to have the same columns as X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [50]:
gp_model = GaussianProcessClassifier(kernel=C(1.0) * RBF(length_scale=1.0), random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
log_reg_model = LogisticRegression(random_state=42)

In [51]:
stacked_model = StackingClassifier(
    estimators=[('gp', gp_model), ('rf', rf_model), ('lr', log_reg_model)],
    final_estimator=LogisticRegression()
)


In [52]:
stacked_model.fit(X_train, y_train)


StackingClassifier(estimators=[('gp',
                                GaussianProcessClassifier(kernel=1**2 * RBF(length_scale=1),
                                                          random_state=42)),
                               ('rf', RandomForestClassifier(random_state=42)),
                               ('lr', LogisticRegression(random_state=42))],
                   final_estimator=LogisticRegression())

In [53]:
# Make predictions using the stacked model
y_pred = stacked_model.predict(X_test)

# Evaluate the performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacked Model Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Stacked Model Accuracy: 0.8612
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.70      0.76       138
           1       0.87      0.94      0.90       287

    accuracy                           0.86       425
   macro avg       0.86      0.82      0.83       425
weighted avg       0.86      0.86      0.86       425

